In [1]:
from spark_utils import get_spark_session, stop_spark_session

# Spark's built-in column functions (trim, casing, date math, conditional logic, etc.)
from pyspark.sql import functions as F

# Start a Spark session
spark = get_spark_session("SilverCleaning")

# Where Bronze tables were written by the previous notebook
BRONZE_DIR = "../delta/bronze"

# Where this notebook will write cleaned Silver tables
SILVER_DIR = "../delta/silver"


D:\Udemy_Python\envs\pyspark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
# Small reusable helper/function: trims whitespace and title-cases a city-name column.
# Used identically on customers, geolocation, and sellers below, so it's defined once here instead of repeated three times.

def clean_city_column(df, column_name):
    return df.withColumn(
        column_name,
        F.initcap(F.trim(F.col(column_name)))   # trim() strips leading/trailing spaces, initcap() title-cases the text
    )


In [3]:
# --- Customers ---

# read raw Bronze table
customers_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_customers_dataset") 

customers_silver = (
    clean_city_column(customers_bronze, "customer_city")   # standardize city casing & whitespace by calling the above function
    .dropDuplicates()                                      # remove any exact duplicate rows
)

# write cleaned Silver table
customers_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/olist_customers_dataset")

print(f"[Silver] customers: {customers_silver.count()} rows written")

[Silver] customers: 99441 rows written


In [4]:
# --- Geolocation ---
# This table has many duplicate/near-duplicate lat-lng rows per zip prefix (it's raw postal survey data). 
# I collapse it down to one averaged lat/lng per zip prefix, which is what downstream joins actually need.

geolocation_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_geolocation_dataset")
# read raw Bronze table

geolocation_silver = (
    clean_city_column(geolocation_bronze, "geolocation_city")   # standardize city casing/whitespace
    .groupBy("geolocation_zip_code_prefix")                     # collapse to one row per zip prefix
    .agg(
        F.avg("geolocation_lat").alias("geolocation_lat"),      # average latitude across duplicate entries
        F.avg("geolocation_lng").alias("geolocation_lng"),      # average longitude across duplicate entries
        F.first("geolocation_city").alias("geolocation_city"),  # take one representative city name
        F.first("geolocation_state").alias("geolocation_state"),# take one representative state code
    )
)

geolocation_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/olist_geolocation_dataset")
# write cleaned Silver table

print(f"[Silver] geolocation: {geolocation_silver.count()} rows written (deduplicated by zip prefix)")

[Silver] geolocation: 19015 rows written (deduplicated by zip prefix)


In [5]:
# --- Order items ---
# Just validate price/freight are non-negative; flag rather than silently drop,
# so we know if bad data exists instead of quietly losing rows.

order_items_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_order_items_dataset")
# read raw Bronze table

# Count how many rows (if any) have a negative price or freight value
invalid_count = order_items_bronze.filter((F.col("price") < 0) | (F.col("freight_value") < 0)).count()
print(f"[Check] order_items rows with negative price/freight: {invalid_count}")

order_items_silver = order_items_bronze  # no rows dropped; this table passes through as-is after the check above

order_items_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/olist_order_items_dataset")
# write Silver table

print(f"[Silver] order_items: {order_items_silver.count()} rows written")

[Check] order_items rows with negative price/freight: 0
[Silver] order_items: 112650 rows written


In [6]:
# --- Order payments ---
# Same light validation as order_items: check for negative payment values.

order_payments_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_order_payments_dataset")
# read raw Bronze table

# count any negative payment amounts
invalid_count = order_payments_bronze.filter(F.col("payment_value") < 0).count()
print(f"[Check] order_payments rows with negative payment_value: {invalid_count}")

order_payments_silver = order_payments_bronze
# passes through as-is after the check

order_payments_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/olist_order_payments_dataset")
# write Silver table

print(f"[Silver] order_payments: {order_payments_silver.count()} rows written")

[Check] order_payments rows with negative payment_value: 0
[Silver] order_payments: 103886 rows written


In [7]:
# --- Order reviews ---
# Most reviews have no written comment - that's a real signal (customer just left a star rating), not missing data.
# I added an explicit has_comment flag instead of leaving it ambiguous, and leave the null text fields as null.

order_reviews_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_order_reviews_dataset")
# read raw Bronze table

order_reviews_silver = order_reviews_bronze.withColumn(
    "has_comment",
    F.when(F.col("review_comment_message").isNotNull(), True).otherwise(False)   # True if a written comment exists, else False
)

order_reviews_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/olist_order_reviews_dataset")
# write Silver table

print(f"[Silver] order_reviews: {order_reviews_silver.count()} rows written")

[Silver] order_reviews: 104162 rows written


In [8]:
# --- Orders ---
# Add two derived columns that KPIs will need directly at the Power BI stage:
#   is_delivered  - simple boolean flag based on order_status
#   delivery_days - days between purchase and delivery, null if never delivered

orders_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_orders_dataset")
# read raw Bronze table

orders_silver = (
    orders_bronze
    .withColumn(
        "is_delivered",
        F.when(F.col("order_status") == "delivered", True).otherwise(False)   # True only for orders actually marked delivered
    )
    .withColumn(
        "delivery_days",
        F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))
        # datediff() returns null automatically if either date is null (e.g. undelivered orders) - no extra handling needed
    )
)

orders_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/olist_orders_dataset")
# write Silver table

print(f"[Silver] orders: {orders_silver.count()} rows written")

[Silver] orders: 99441 rows written


In [9]:
# --- Category translation (passthrough) ---
# Small lookup table, no cleaning needed - just copied to Silver so every
# downstream notebook reads it from one consistent location.

category_translation_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/product_category_name_translation")
# read raw Bronze table

category_translation_silver = category_translation_bronze
# no changes needed

category_translation_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/product_category_name_translation")
# write Silver table

print(f"[Silver] category_translation: {category_translation_silver.count()} rows written")

[Silver] category_translation: 71 rows written


In [10]:
# --- Products ---
# Left-join the category translation table to add an English category name.
# Where no translation match exists (or the category itself is null), fill
# with "unknown" instead of leaving it null, so downstream group-bys don't
# silently drop those rows.

products_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_products_dataset")
# read raw Bronze table

products_silver = (
    products_bronze
    .join(
        category_translation_silver,                # the lookup table just cleaned above
        on="product_category_name",                 # join key shared by both tables
        how="left"                                  # keep all products even if no translation match exists
    )
    .withColumn(
        "product_category_name_english",
        F.coalesce(F.col("product_category_name_english"), F.lit("unknown"))   # fill nulls (no match, or original was null) with "unknown"
    )
)

products_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/olist_products_dataset")
# write Silver table

print(f"[Silver] products: {products_silver.count()} rows written")

[Silver] products: 32951 rows written


In [11]:
# --- Sellers ---
# Same city standardization as customers; drop exact duplicates if any.

sellers_bronze = spark.read.format("delta").load(f"{BRONZE_DIR}/olist_sellers_dataset")
# read raw Bronze table

sellers_silver = (
    clean_city_column(sellers_bronze, "seller_city")   # standardizing city casing/whitespace
    .dropDuplicates()                                  # remove any exact duplicate rows
)

sellers_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/olist_sellers_dataset")
# write Silver table

print(f"[Silver] sellers: {sellers_silver.count()} rows written")

[Silver] sellers: 3095 rows written


In [12]:
# Quick sanity check: read the cleaned orders Silver table back and inspect
# the two derived columns I added, to confirm they look right.

check_df = spark.read.format("delta").load(f"{SILVER_DIR}/olist_orders_dataset")
check_df.select("order_id", "order_status", "is_delivered", "delivery_days").show(10, truncate=False)

+--------------------------------+------------+------------+-------------+
|order_id                        |order_status|is_delivered|delivery_days|
+--------------------------------+------------+------------+-------------+
|d9eabc69f974b3d088e991cb7a7f08cd|delivered   |true        |7            |
|c9fbf88f9c58364e0434971962dc1a53|delivered   |true        |15           |
|fd01a48a7d75383a3623e71a538584c3|delivered   |true        |10           |
|6d4616de4341417e17978fe57aec1c46|delivered   |true        |20           |
|c606769bddf9fb8b9f949b128133c463|delivered   |true        |8            |
|5ef95d0a45b4caaaa70e601665ba1a6a|delivered   |true        |28           |
|9f2fe2350791b5e9e62610331d125843|delivered   |true        |7            |
|86bf07c0a6d01d38dd8ecd1a1b68d139|delivered   |true        |9            |
|2eb455a9ac809470168098ddf4e2a25b|delivered   |true        |4            |
|5e4d2d8a6589ec90d1c82d0c2cb55eb0|delivered   |true        |4            |
+------------------------

In [13]:
stop_spark_session(spark)